# From Microscope to GPU: Bioimage Analysis with Cellpose

**Workshop — WASP/DDLS (Google Colab Version)**

> End-to-end pipeline: download a public microscopy dataset, run
> GPU-accelerated Cellpose segmentation, extract per-cell measurements,
> and visualize results — all inside a free Colab notebook.

| Component | Detail |
|-----------|--------|
| **Dataset** | BBBC020 — murine bone-marrow macrophages (25 image sets, 2 channels) |
| **Segmentation** | Cellpose `cyto3` model on GPU |
| **Runtime** | Google Colab (free T4 GPU) |

---

### Pipeline Overview

```
  Broad Institute              Colab Runtime
 ┌──────────────┐        ┌──────────────────────┐
 │  BBBC020 zip │──────> │  /content/            │
 └──────────────┘ wget   │  BBBC020_v1_images/   │
                         └──────────┬───────────┘
                                    │
                          Step 1-2  │ Explore images
                          Step 3    │ Cellpose GPU (single demo)
                          Step 4    │ Cellpose GPU (full batch)
                                    │
                         ┌──────────v───────────┐
                         │  /content/results/    │
                         │  masks + measurements │
                         └──────────┬───────────┘
                                    │
                          Step 5    │ Visualize & analyze
                                    v
                         Charts, tables, statistics
```

> **Note:** The full workshop version of this notebook also integrates with
> [OMERO](https://omero.readthedocs.io) for image management and result
> push-back. This Colab version is self-contained and does not require
> OMERO or HPC access.

---
## Step 0 — Install Packages

Install the Python packages required for this notebook. This takes about a
minute on a fresh Colab runtime.

In [ ]:
!pip install -q cellpose>=3.0 tifffile scikit-image pandas numpy matplotlib ipywidgets tqdm

---
## Step 1 — Environment Check

Make sure the GPU is visible and all required packages are installed.

> **Tip:** If no GPU is shown below, go to *Runtime → Change runtime type*
> and select a **T4 GPU**.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader

In [ ]:
import importlib, sys

packages = {
    "cellpose":   "cellpose",
    "tifffile":   "tifffile",
    "numpy":      "numpy",
    "pandas":     "pandas",
    "matplotlib": "matplotlib",
    "skimage":    "scikit-image"
}

print(f"Python {sys.version}\n")
for mod, pkg in packages.items():
    try:
        m = importlib.import_module(mod.split(".")[0])
        ver = getattr(m, "__version__", "installed")
        print(f"  {pkg:20s} {ver}")
    except ImportError:
        print(f"  {pkg:20s} ** NOT FOUND **")

try:
    import torch
    print(f"\n  PyTorch {torch.__version__}  |  CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"  GPU device: {torch.cuda.get_device_name(0)}")
except ImportError:
    print("\n  PyTorch not installed (Cellpose will use its own GPU backend)")

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tifffile import imread, imwrite

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

---
## Configuration

In [ ]:
from pathlib import Path

DATA_DIR    = Path("/content/BBBC020_v1_images")
RESULTS_DIR = Path("/content/results")

CELLPOSE_MODEL = "cyto3"

---
## Download the BBBC020 Dataset

[BBBC020](https://bbbc.broadinstitute.org/BBBC020) is a public dataset from
the Broad Bioimage Benchmark Collection. We download it directly.

In [ ]:
import os, zipfile, urllib.request

DATASET_URL = "https://data.broadinstitute.org/bbbc/BBBC020/BBBC020_v1_images.zip"
ZIP_PATH    = "/content/BBBC020_v1_images.zip"

if not DATA_DIR.exists():
    print(f"Downloading BBBC020 dataset ...")
    urllib.request.urlretrieve(DATASET_URL, ZIP_PATH)
    print(f"Extracting ...")
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall("/content")
    os.remove(ZIP_PATH)
    print(f"Done — dataset at {DATA_DIR}")
else:
    print(f"Dataset already present at {DATA_DIR}")

---
## Utility Functions

These helper functions handle BBBC020 image discovery, channel loading,
measurement extraction, and result saving. They are adapted from the
workshop's `utils.py`.

In [ ]:
import re
from typing import Optional
from skimage.measure import regionprops_table

NUCLEI_PATTERN    = re.compile(r"_c1\.TIF$", re.IGNORECASE)
BODY_PATTERN      = re.compile(r"_c5\.TIF$", re.IGNORECASE)
COMPOSITE_PATTERN = re.compile(r"_\(c1\+c5\)\.TIF$", re.IGNORECASE)


def _parse_condition(name: str) -> tuple[str, Optional[int]]:
    """Extract condition and replicate from a folder name like 'jw-15min 3'."""
    m = re.match(r"jw-(\S+?)(\d*)$", name.replace(" ", ""))
    if not m:
        return name, None
    cond = m.group(1).rstrip("0123456789")
    rep_str = name.split()[-1] if " " in name else m.group(2)
    try:
        rep = int(rep_str)
    except (ValueError, TypeError):
        rep = None
    return cond, rep


def discover_image_pairs(data_dir: str | Path) -> list[dict]:
    """Find paired nuclei/body channel TIFs under *data_dir*.

    Returns a list of dicts with keys:
        name, nuclei, body, composite, condition, replicate
    """
    data_dir = Path(data_dir)
    pairs: dict[str, dict] = {}

    for tif in sorted(data_dir.rglob("*.TIF")):
        name = tif.parent.name
        if name not in pairs:
            pairs[name] = {
                "name": name,
                "nuclei": None,
                "body": None,
                "composite": None,
            }

        if NUCLEI_PATTERN.search(tif.name):
            pairs[name]["nuclei"] = tif
        elif BODY_PATTERN.search(tif.name):
            pairs[name]["body"] = tif
        elif COMPOSITE_PATTERN.search(tif.name):
            pairs[name]["composite"] = tif

    result = []
    for p in pairs.values():
        condition, replicate = _parse_condition(p["name"])
        p["condition"] = condition
        p["replicate"] = replicate
        result.append(p)

    result.sort(key=lambda x: (x["condition"], x["replicate"] or 0))
    return result


def load_channels(pair: dict) -> np.ndarray:
    """Load nuclei + body channels and stack to (Y, X, 2) for Cellpose.

    Channel order: [body/cytoplasm, nuclei] matching Cellpose channels=[1, 2].
    """
    nuc = imread(str(pair["nuclei"]))
    body = imread(str(pair["body"]))

    if nuc.ndim == 3:
        nuc = nuc.max(axis=-1) if nuc.shape[2] <= 4 else nuc[0]
    if body.ndim == 3:
        body = body.max(axis=-1) if body.shape[2] <= 4 else body[0]

    return np.stack([body, nuc], axis=-1)


def extract_measurements(masks: np.ndarray,
                         intensity_image: np.ndarray) -> pd.DataFrame:
    """Extract per-cell measurements from a label mask."""
    if intensity_image.ndim == 3:
        intensity_image = intensity_image.max(axis=-1)

    props = regionprops_table(
        masks,
        intensity_image=intensity_image,
        properties=[
            "label", "area", "centroid",
            "mean_intensity", "eccentricity",
        ],
    )
    return pd.DataFrame(props)


def save_results(masks: np.ndarray,
                 measurements: pd.DataFrame,
                 out_dir: str | Path,
                 stem: str) -> tuple[Path, Path]:
    """Save mask TIFF and measurements CSV. Returns (mask_path, csv_path)."""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    mask_path = out_dir / f"{stem}_masks.tiff"
    csv_path  = out_dir / f"{stem}_measurements.csv"

    imwrite(str(mask_path), masks.astype(np.uint16))
    measurements.to_csv(csv_path, index=False)

    return mask_path, csv_path

---
## Step 2 — Explore the BBBC020 Dataset

[BBBC020](https://bbbc.broadinstitute.org/BBBC020) contains fluorescence images of murine bone-marrow derived macrophages treated with *Wiskostatin* at different timepoints.

Each image set has two channels:
- `_c1.TIF` — **Hoechst 33342** (nuclei)
- `_c5.TIF` — **Phalloidin-TRITC** (actin / cell body)

Let's discover all 25 image pairs and preview a few.

In [ ]:
pairs = discover_image_pairs(DATA_DIR)
print(f"Found {len(pairs)} image sets\n")

summary = pd.DataFrame([
    {"Condition": p["condition"], "Replicate": p["replicate"], "Name": p["name"]}
    for p in pairs
])
summary.groupby("Condition").size().rename("Count").to_frame()

In [ ]:
samples = [pairs[0], pairs[10], pairs[-1]]

fig, axes = plt.subplots(3, 3, figsize=(14, 12))
col_labels = ["Nuclei (Hoechst)", "Cell body (Phalloidin)", "Composite"]

for row, pair in enumerate(samples):
    imgs = [
        imread(str(pair["nuclei"])),
        imread(str(pair["body"])),
        imread(str(pair["composite"])) if pair["composite"] else np.zeros_like(imread(str(pair["nuclei"]))),
    ]
    cmaps = ["Blues", "Oranges", None]
    for col, (img, cmap) in enumerate(zip(imgs, cmaps)):
        ax = axes[row, col]
        if img.ndim == 3 and col < 2:
            img = img.max(axis=-1) if img.shape[2] <= 4 else img[0]
        ax.imshow(img, cmap=cmap)
        ax.set_title(f"{pair['name']}\n{col_labels[col]}" if row == 0 else col_labels[col],
                     fontsize=10)
        ax.axis("off")

fig.suptitle("BBBC020 Image Gallery — Sample Fields of View", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

---
## Step 3 — GPU Segmentation: Single Image Demo

Before running the full batch, let's process **one image pair** interactively
so we can inspect the segmentation quality and see what Cellpose produces.

In [ ]:
import time
from cellpose import models

demo_pair = pairs[0]
print(f"Demo image: {demo_pair['name']}")

In [ ]:
nuc = imread(str(demo_pair["nuclei"]))
if nuc.ndim == 3:
    nuc = nuc.max(axis=-1) if nuc.shape[2] <= 4 else nuc[0]

has_body = demo_pair["body"] is not None
if has_body:
    body = imread(str(demo_pair["body"]))
    if body.ndim == 3:
        body = body.max(axis=-1) if body.shape[2] <= 4 else body[0]
    img_2ch = np.stack([body, nuc], axis=-1)
    channels = [1, 2]
else:
    img_2ch = nuc
    channels = [0, 0]

fig, axes = plt.subplots(1, 2 if has_body else 1, figsize=(12, 5))
if not has_body:
    axes = [axes]
axes[0].imshow(nuc, cmap="Blues")
axes[0].set_title("Nuclei (Hoechst)")
axes[0].axis("off")
if has_body:
    axes[1].imshow(body, cmap="Oranges")
    axes[1].set_title("Cell body (Phalloidin)")
    axes[1].axis("off")
plt.suptitle(f"Raw channels — {demo_pair['name']}", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
print(f"Loading Cellpose '{CELLPOSE_MODEL}' model (GPU) ...")
model = models.CellposeModel(model_type=CELLPOSE_MODEL, gpu=True)

t0 = time.time()
masks, flows, styles = model.eval(
    img_2ch,
    diameter=None,
    channels=channels,
    flow_threshold=0.4,
    cellprob_threshold=0.0,
)
elapsed = time.time() - t0

n_cells = int(masks.max())
print(f"\nSegmented {n_cells} cells in {elapsed:.1f}s")

In [ ]:
from matplotlib.colors import ListedColormap

np.random.seed(42)
colors = np.random.rand(n_cells + 1, 4)
colors[:, 3] = 0.5
colors[0] = [0, 0, 0, 0]
label_cmap = ListedColormap(colors)

ncols = 3 if has_body else 2
fig, axes = plt.subplots(1, ncols, figsize=(6 * ncols, 6))

axes[0].imshow(nuc, cmap="Blues")
axes[0].set_title("Nuclei")
axes[0].axis("off")

col = 1
if has_body:
    axes[col].imshow(body, cmap="Oranges")
    axes[col].set_title("Cell body")
    axes[col].axis("off")
    col += 1

axes[col].imshow(nuc, cmap="gray")
axes[col].imshow(masks, cmap=label_cmap, interpolation="nearest")
axes[col].set_title(f"Cellpose mask ({n_cells} cells)")
axes[col].axis("off")

fig.suptitle(f"Segmentation result — {demo_pair['name']}  ({elapsed:.1f}s on GPU)",
             fontsize=13)
plt.tight_layout()
plt.show()

---
## Step 4 — Batch Segmentation (All Images)

Now we process **every image pair** in the dataset. The model is already loaded
in GPU memory, so this should be fast.

Results (masks + per-cell measurements) are saved to `results/`.

In [ ]:
from tqdm.notebook import tqdm

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

results_rows = []
total_t0 = time.time()

for pair in tqdm(pairs, desc="Segmenting"):
    nuc_img = imread(str(pair["nuclei"]))
    if nuc_img.ndim == 3:
        nuc_img = nuc_img.max(axis=-1) if nuc_img.shape[2] <= 4 else nuc_img[0]

    if pair["body"] is not None:
        body_img = imread(str(pair["body"]))
        if body_img.ndim == 3:
            body_img = body_img.max(axis=-1) if body_img.shape[2] <= 4 else body_img[0]
        img = np.stack([body_img, nuc_img], axis=-1)
        ch = [1, 2]
    else:
        img = nuc_img
        ch = [0, 0]

    t0 = time.time()
    m, _, _ = model.eval(img, diameter=None, channels=ch,
                         flow_threshold=0.4, cellprob_threshold=0.0)
    seg_time = time.time() - t0

    nc = int(m.max())
    intensity = img if img.ndim == 2 else img[..., 1]
    meas = extract_measurements(m, intensity)
    save_results(m, meas, RESULTS_DIR, pair["name"])

    results_rows.append({
        "name": pair["name"],
        "n_cells": nc,
        "seg_time_s": round(seg_time, 2),
    })

total_time = time.time() - total_t0

summary_df = pd.DataFrame(results_rows)
summary_df.to_csv(RESULTS_DIR / "summary.csv", index=False)

total_cells = summary_df["n_cells"].sum()
print(f"\nDone: {total_cells} total cells across {len(pairs)} images "
      f"in {total_time:.1f}s")
print(f"Results saved to: {RESULTS_DIR}")

---
## Step 5 — Analyze and Visualize Results

Let's look at the segmentation results across all conditions and timepoints.

In [ ]:
summary_df = pd.read_csv(RESULTS_DIR / "summary.csv")

def extract_condition(name):
    """Extract condition (e.g. '15min', 'Kontrolle') from image name."""
    m = re.match(r"jw-(\S+?)(\s*\d*)$", name)
    if not m:
        return name
    cond = m.group(1).rstrip("0123456789")
    return cond

summary_df["condition"] = summary_df["name"].apply(extract_condition)
summary_df

In [ ]:
condition_order = ["15min", "30min", "1h", "2h", "24h", "Kontrolle"]
palette = dict(zip(condition_order, plt.cm.tab10.colors))

fig, ax = plt.subplots(figsize=(14, 5))
bars = ax.bar(
    range(len(summary_df)),
    summary_df["n_cells"],
    color=[palette.get(c, "gray") for c in summary_df["condition"]],
    edgecolor="white",
    linewidth=0.5,
)
ax.set_xticks(range(len(summary_df)))
ax.set_xticklabels(summary_df["name"], rotation=45, ha="right", fontsize=8)
ax.set_ylabel("Cell count")
ax.set_title("Cells detected per image")

from matplotlib.patches import Patch
handles = [Patch(facecolor=palette.get(c, "gray"), label=c) for c in condition_order
           if c in summary_df["condition"].values]
ax.legend(handles=handles, title="Condition", loc="upper right")
plt.tight_layout()
plt.show()

In [ ]:
all_measurements = []
for _, row in summary_df.iterrows():
    csv_path = RESULTS_DIR / f"{row['name']}_measurements.csv"
    if csv_path.exists():
        df = pd.read_csv(csv_path)
        df["image"] = row["name"]
        df["condition"] = row["condition"]
        all_measurements.append(df)

all_meas = pd.concat(all_measurements, ignore_index=True)
print(f"Total cells measured: {len(all_meas)}")
all_meas.head()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

cond_present = [c for c in condition_order if c in all_meas["condition"].values]
data_by_cond = [all_meas[all_meas["condition"] == c]["area"].values
                for c in cond_present]

bp = ax.boxplot(data_by_cond, labels=cond_present, patch_artist=True,
                showfliers=False, widths=0.6)
for patch, cond in zip(bp["boxes"], cond_present):
    patch.set_facecolor(palette.get(cond, "gray"))
    patch.set_alpha(0.7)

ax.set_ylabel("Cell area (pixels)")
ax.set_xlabel("Condition")
ax.set_title("Cell area distribution across conditions")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for cond in cond_present:
    subset = all_meas[all_meas["condition"] == cond]
    ax.scatter(
        subset["mean_intensity"],
        subset["eccentricity"],
        c=[palette.get(cond, "gray")],
        label=cond,
        alpha=0.4,
        s=15,
        edgecolors="none",
    )

ax.set_xlabel("Mean intensity")
ax.set_ylabel("Eccentricity")
ax.set_title("Cell morphology: intensity vs. shape")
ax.legend(title="Condition", markerscale=3)
plt.tight_layout()
plt.show()

In [ ]:
stats = (
    all_meas.groupby("condition")
    .agg(
        total_cells=("label", "count"),
        mean_area=("area", "mean"),
        std_area=("area", "std"),
        mean_intensity=("mean_intensity", "mean"),
        mean_eccentricity=("eccentricity", "mean"),
    )
    .reindex(cond_present)
    .round(1)
)
stats

**Question for the audience:** Do you see differences in cell area or shape between the treatment timepoints and the control (Kontrolle)? What biological process might explain this?

---
## Summary

In this notebook we walked through a complete **bioimage analysis pipeline**:

| Step | What we did |
|------|-------------|
| **0. Setup** | Installed packages and verified GPU availability on Colab |
| **1. Environment** | Checked GPU, Python packages, and CUDA |
| **2. Explore** | Downloaded BBBC020 and discovered 25 image sets across 6 conditions |
| **3. Single demo** | Ran Cellpose on one image, inspected the segmentation mask |
| **4. Batch** | Segmented all 25 images on GPU with a progress bar |
| **5. Analysis** | Visualized cell counts, area distributions, and morphology across conditions |

### Full workshop pipeline

In the full workshop version of this notebook (running on the ALVIS HPC
cluster), two additional steps integrate with **OMERO** for image management:

- **Connect to OMERO** — resolve NFS file paths via the OMERO API for
  zero-copy GPU access on the cluster
- **Push results back** — attach masks, measurement CSVs, and polygon ROIs
  to OMERO images so collaborators can view them in a web browser

---

### Resources

| Resource | Link |
|----------|------|
| SciLifeLab OMERO | https://omero.scilifelab.se |
| OMERO documentation | https://omero.readthedocs.io |
| Cellpose docs | https://cellpose.readthedocs.io |
| BBBC020 dataset | https://bbbc.broadinstitute.org/BBBC020 |

---

### Questions?

Thank you for participating!